# 试题3：使用MindSpore搭建Unet类，用于构建网络。要求：网络结构和原论文保持一致。

U-Net网络结构说明：
- U-Net是一种用于生物医学图像分割的全卷积神经网络
- 网络结构呈U形，包含编码器（下采样路径）和解码器（上采样路径）
- 编码器通过连续的卷积和池化操作提取特征
- 解码器通过上采样和跳跃连接（skip connections）恢复空间分辨率
- 跳跃连接将编码器的特征图与解码器的特征图拼接，融合不同层次的特征

原论文网络结构：
- 输入尺寸: 572x572
- 输出尺寸: 388x388
- 每次卷积使用3x3的卷积核，无padding
- 使用最大池化进行下采样
- 使用转置卷积进行上采样
- 使用跳跃连接（skip connections）融合特征

In [ ]:
import mindspore.nn as nn
import mindspore.ops.operations as F
from mindspore.common.initializer import TruncatedNormal
from mindspore.dataset.vision import c_transforms as c_vision

## 定义网络模块

In [ ]:
class DoubleConv(nn.Cell):
    """
    双重卷积模块
    包含两个连续的3x3卷积层，每个卷积层后接ReLU激活函数
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        mid_channels: 中间通道数（可选，默认为out_channels）
    """
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        # 截断高斯分布初始化
        init_value_0 = TruncatedNormal(0.06)
        init_value_1 = TruncatedNormal(0.06)
        
        if not mid_channels:
            mid_channels = out_channels
        
        # 定义两个卷积层
        # 根据原论文，激活函数用ReLU，卷积核大小为3，不用padding
        # 因此每次卷积之后feature map的尺寸会减小2像素
        self.double_conv = nn.SequentialCell([
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, has_bias=True, 
                     weight_init=init_value_0, pad_mode="valid"),
            nn.ReLU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, has_bias=True, 
                     weight_init=init_value_1, pad_mode="valid"),
            nn.ReLU()
        ])
    
    def construct(self, x):
        return self.double_conv(x)

class Down(nn.Cell):
    """
    下采样模块
    包含一个最大池化层和双重卷积模块
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 最大池化（2x2，步长为2）+ 双重卷积
        self.maxpool_conv = nn.SequentialCell([
            nn.MaxPool2d(kernel_size=2, stride=2),
            DoubleConv(in_channels, out_channels)
        ])
    
    def construct(self, x):
        return self.maxpool_conv(x)

class Up1(nn.Cell):
    """
    上采样模块1
    用于解码器的第一个上采样层
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        bilinear: 是否使用双线性插值（本实现使用转置卷积）
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        # U-Net网络需要拼接编码器和解码器的特征图
        self.concat = F.Concat(axis=1)
        
        # 根据原论文，编码器中的特征图经过中心裁剪后，进行拼接
        # 编码器的特征图大小为64，解码器为56
        self.factor = 56.0 / 64.0
        self.center_crop = c_vision.CentralCrop(central_fraction=self.factor)
        
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        
        # 转置卷积，常用的上采样方式
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, 
                                    kernel_size=2, stride=2)
        self.relu = nn.ReLU()
    
    def construct(self, x1, x2):
        # 上采样
        x1 = self.up(x1)
        x1 = self.relu(x1)
        
        # 中心裁剪编码器的特征图
        x2 = self.center_crop(x2)
        
        # 拼接特征图
        x = self.concat((x1, x2))
        
        return self.conv(x)

class Up2(nn.Cell):
    """
    上采样模块2
    用于解码器的第二个上采样层
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        
        # 编码器特征图大小为136，解码器为104
        self.factor = 104.0 / 136.0
        self.center_crop = c_vision.CentralCrop(central_fraction=self.factor)
        
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, 
                                    kernel_size=2, stride=2)
        self.relu = nn.ReLU()
    
    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)

class Up3(nn.Cell):
    """
    上采样模块3
    用于解码器的第三个上采样层
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        
        # 编码器特征图大小为280，解码器为200
        self.factor = 200 / 280
        self.center_crop = c_vision.CentralCrop(central_fraction=self.factor)
        
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, 
                                    kernel_size=2, stride=2)
        self.relu = nn.ReLU()
    
    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)

class Up4(nn.Cell):
    """
    上采样模块4
    用于解码器的第四个上采样层
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        
        # 编码器特征图大小为568，解码器为392
        self.factor = 392 / 568
        self.center_crop = c_vision.CentralCrop(central_fraction=self.factor)
        
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, 
                                    kernel_size=2, stride=2)
        self.relu = nn.ReLU()
    
    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)

class OutConv(nn.Cell):
    """
    输出卷积层
    将特征图转换为最终的分割结果
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数（类别数）
    """
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        init_value = TruncatedNormal(0.06)
        # 1x1卷积，用于调整通道数
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                             has_bias=True, weight_init=init_value)
    
    def construct(self, x):
        x = self.conv(x)
        return x

## 定义U-Net网络

In [ ]:
class UNet(nn.Cell):
    """
    U-Net网络
    
    完整的U-Net网络结构，包含编码器和解码器
    
    参数:
        n_channels: 输入图像的通道数（灰度图为1，RGB图为3）
        n_classes: 输出的类别数（二分类为2）
    
    网络结构:
        - 编码器（下采样路径）:
          * inc: DoubleConv(1 -> 64)
          * down1: Down(64 -> 128)
          * down2: Down(128 -> 256)
          * down3: Down(256 -> 512)
          * down4: Down(512 -> 1024)
        
        - 解码器（上采样路径）:
          * up1: Up1(1024 -> 512)
          * up2: Up2(512 -> 256)
          * up3: Up3(256 -> 128)
          * up4: Up4(128 -> 64)
        
        - 输出层:
          * outc: OutConv(64 -> n_classes)
    """
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        
        # 编码器部分
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        
        # 解码器部分
        self.up1 = Up1(1024, 512)
        self.up2 = Up2(512, 256)
        self.up3 = Up3(256, 128)
        self.up4 = Up4(128, 64)
        
        # 输出层
        self.outc = OutConv(64, n_classes)
    
    def construct(self, x):
        """
        前向传播
        
        参数:
            x: 输入图像张量，形状为 (batch_size, n_channels, height, width)
        
        返回:
            logits: 输出logits，形状为 (batch_size, n_classes, height-184, width-184)
                    对于572x572的输入，输出为388x388
        """
        # 编码器路径（下采样）
        x1 = self.inc(x)       # 572 -> 568,  1 -> 64
        x2 = self.down1(x1)    # 568 -> 284 -> 280, 64 -> 128
        x3 = self.down2(x2)    # 280 -> 140 -> 136, 128 -> 256
        x4 = self.down3(x3)    # 136 -> 68 -> 64, 256 -> 512
        x5 = self.down4(x4)    # 64 -> 32 -> 28, 512 -> 1024
        
        # 解码器路径（上采样）+ 跳跃连接
        x = self.up1(x5, x4)   # 28 -> 56 -> 104, 1024 -> 512
        x = self.up2(x, x3)    # 104 -> 208 -> 200, 512 -> 256
        x = self.up3(x, x2)    # 200 -> 400 -> 392, 256 -> 128
        x = self.up4(x, x1)    # 392 -> 784 -> 776, 128 -> 64
        
        # 输出层
        logits = self.outc(x)   # 776 -> 776 (1x1卷积不改变尺寸)
        
        return logits

## 定义网络摘要函数

In [ ]:
def print_network_summary(model, input_size=(1, 1, 572, 572)):
    """
    打印网络结构摘要
    
    参数:
        model: UNet模型实例
        input_size: 输入张量大小
    """
    print("="*80)
    print("U-Net网络结构摘要")
    print("="*80)
    print(f"输入尺寸: {input_size}")
    print(f"输出类别数: {model.n_classes}")
    print(f"输入通道数: {model.n_channels}")
    print("\n网络结构:")
    print("  编码器（下采样路径）:")
    print("    - inc: DoubleConv(1 -> 64)")
    print("    - down1: Down(64 -> 128)")
    print("    - down2: Down(128 -> 256)")
    print("    - down3: Down(256 -> 512)")
    print("    - down4: Down(512 -> 1024)")
    print("\n  解码器（上采样路径）:")
    print("    - up1: Up1(1024 -> 512) [跳跃连接: x4]")
    print("    - up2: Up2(512 -> 256) [跳跃连接: x3]")
    print("    - up3: Up3(256 -> 128) [跳跃连接: x2]")
    print("    - up4: Up4(128 -> 64) [跳跃连接: x1]")
    print("\n  输出层:")
    print("    - outc: OutConv(64 -> 2)")
    print("\n网络特点:")
    print("  - 全卷积网络，无全连接层")
    print("  - 使用跳跃连接（skip connections）融合多尺度特征")
    print("  - 编码器提取特征，解码器恢复空间分辨率")
    print("  - 输入尺寸大于输出尺寸（572x572 -> 388x388）")
    print("="*80)

## 创建和测试模型

In [ ]:
# 创建U-Net模型
print("创建U-Net模型...")
model = UNet(n_channels=1, n_classes=2)

# 打印网络摘要
print_network_summary(model)

## 测试前向传播

In [ ]:
import numpy as np
from mindspore import Tensor

print("\n测试前向传播...")
# 创建随机输入张量
input_tensor = Tensor(np.random.randn(2, 1, 572, 572).astype(np.float32))
print(f"输入张量形状: {input_tensor.shape}")

# 前向传播
output = model(input_tensor)
print(f"输出张量形状: {output.shape}")

# 计算参数量
total_params = sum([np.prod(p.shape) for p in model.trainable_params()])
print(f"\n网络参数总数: {total_params:,}")

print("\nU-Net网络搭建完成！")